# 01 — Feature characterization audit (Gate 0a)

**The pre-registered protocol that replaces reactive per-feature fixes** (frequency-ensemble spec
§2.5). Every one of the 12 inputs (8 continuous + the EFG block) passes the same univariate audit;
rules R1–R4 (constants frozen in `config.AUDIT` *before* this ran) assign each feature a
transform, a valuation class, a value lever, and — where earned — a derived target. Spec
**v0.8** additions: transform screening is **universal** ({identity, log1p, sqrt} for every
feature + flip/reciprocal where cost-oriented; adoption gated by R1's value-model test with
the concavity distinction — a concave transform asserts diminishing value *per cell*, a
target asserts it *at the portfolio level*), and the audit renders **feature cards** — one
6-panel page per input + a summary sheet — for review against the frozen rules **before**
any Gate-0 solve. The current
formulation is then **re-derived** from the protocol rather than asserted: agreements validate the
rules, disagreements are findings and are **adopted**.

**Zero solves.** Reads the finished hand-off stack, writes the audit archive to `audit/audit_objects/`, the
feature cards to `audit/feature_cards/`, and F8 to `figures/`.

**Gate-0 campaign order** (each notebook top-to-bottom once):

| step | notebook | kernel |
|---|---|---|
| 1 | **this one** — T2 table, F8 figure, archive; prints the lever-ready arm dicts | `y2y-geo` |
| 2 | `02_solve.ipynb` × 4–5 — one arm per pass (a0…a3, optional a4) -- a0 fast; target-bearing arms took **~71 min** on the measured w=1 run, so budget an afternoon | `y2y-r` |
| 3 | `03_gate0_validation.ipynb` — the Gate 0 verdict tables → report-back | `y2y-geo` |

**Expected classifications** (pre-checked; the audit must confirm or the surprise is the finding):
mineral-soil carbon → concentrated-satiating (target 0.332); biomass → **reverts to
diffuse-linear** by the R2 tail-mass criterion (implied target 0.066 < t_min); intactness → R3
inexpressible; connectivity / macrorefugia / corridors / AOH → diffuse-linear; ~36/40 EFGs →
rare-attainable. Watch item (spec §2.5c): AOH birds.

**Kernel:** `Python (y2y-geo)`. Ethan runs; Claude never executes cells.

In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/y2y/, below the repo root where config.py and the engines sit.
import sys, pathlib
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import numpy as np
import pandas as pd

import config, leverage_core as lc
importlib.reload(config); importlib.reload(lc)

HERE = ROOT / "analyses" / "y2y"
SPEC, FIGS = HERE / "spec", HERE / "figures"
AUDIT_OBJ, CARDS = HERE / "audit" / "audit_objects", HERE / "audit" / "feature_cards"
for _d in (FIGS, AUDIT_OBJ, CARDS):
    _d.mkdir(parents=True, exist_ok=True)

# The frozen R4 constants -- printed so the run record shows exactly what this audit was run
# under. Changing any of these is a spec revision, not a notebook edit.
print("R4 constants (FROZEN before the audit ran):")
for k, v in config.AUDIT.items():
    print(f"  {k:<14} {v}")
print(f"  budget_pct     {config.BUDGET_PCT}   (audit is CONDITIONAL on this budget -- spec s2.5 note a)")

## R1 — universal transform screening (v0.8)

Every feature, every candidate transform — leverage *and* the classification it would produce.
Screening is universal; **adoption is gated by the value-model test**, never by leverage. The
concavity distinction is binding: log/sqrt assert diminishing value *per cell density* (false for
carbon — a tonne is a tonne; the open case for AOH richness, identity retained for paper 1), while
a **target** asserts diminishing value *at the portfolio level*. Rank/percentile stretches remain
inadmissible.

In [ ]:
# ---- R1: UNIVERSAL transform screening (v0.8) -------------------------------
# Every feature is screened under {identity, log1p, sqrt} (+ flip/reciprocal where the layer is
# cost-oriented); leverage AND resulting classification are reported per transform. ADOPTION is
# gated by R1's value-model test, never by leverage improvement -- note how log1p would FLIP
# m_soc to diffuse-linear (destroying the tail that earns its target: the E9 log-arm story) and
# would push AOH birds below the expressivity floor.
for name in lc.continuous_features():
    print(f"{name}:")
    for label, d in lc.transform_response(name).items():
        print(f"    {label:<28} lev={d['leverage']:.3f}  {d['cls']:<28} "
              f"{'ADMISSIBLE' if d['admissible'] else 'screened only'}")
    print()

## T2 — the characterization table (rules R1–R4 applied)

One classified row per continuous feature, plus the EFG block with its unsaturated-minority
disclosure. Only the two **pre-checked** classifications are asserted; everything else prints —
a surprise here is a finding for the report-back, not a failure.

In [ ]:
# ---- Run the audit + apply the rules -> T2 ----------------------------------
tbl, efg_caps = lc.characterization_table()

# Assert ONLY what was pre-checked this session (plan gate G-0a-code); print the rest.
row = tbl.set_index("feature")
assert row.loc["irrecoverable_carbon_m_soc", "cls"] == "concentrated-satiating", \
    "m_soc no longer classifies concentrated-satiating -- the stack changed; STOP and diagnose"
assert abs(row.loc["irrecoverable_carbon_m_soc", "target"] - 0.332) < 0.005, \
    f"m_soc derived target moved: {row.loc['irrecoverable_carbon_m_soc','target']} (expected 0.332)"
assert row.loc["irrecoverable_carbon_biomass", "cls"] == "diffuse-linear", \
    "biomass did NOT revert by the tail-mass criterion -- the R2 expectation failed; report this"

print("\nasserted: m_soc = concentrated-satiating (target 0.332); biomass = diffuse-linear (reverted by t_min)")
print("\nnotes for the report-back:")
mac = row.loc["climate_type_macrorefugia"]
print(f"  - macrorefugia theta-crossing at {100*mac.theta_area:.2f}% area is a NEAR-MISS of a_min "
      f"(0.5%) -- but it also fails t_min ({mac.theta_target:.3f} < {config.AUDIT['t_min']}), so its "
      f"diffuse-linear class is robust to either leg.")
birds = row.loc["aoh_richness_birds"]
print(f"  - watch item AOH birds: theta_area = {birds.theta_area:.4f} -> "
      f"{'NO concentrated tail; stays weight-levered' if birds.theta_area < config.AUDIT['a_min'] else 'HAS a tail -- documented decision needed'}")
print(f"  - mammals at leverage {row.loc['aoh_richness_mammals','leverage']:.3f} is the marginally-live flag R4 predicts.")

## F8 — marginal-density trajectories (the protocol's key figure)

Why mineral soil earns a stopping rule and biomass does not, with no equations: mineral soil rides
above θ for a long shelf (≥5× the regional mean sustained over ~4% of the region); biomass crosses
early with almost no mass behind it; the flat layers never reach θ at all.

In [ ]:
# ---- F8 -> analyses/y2y/figures/ ---------------------------------------------
fig = lc.trajectory_figure(FIGS / "F8_marginal_density_trajectories.png")

## Feature cards (v0.8) — review BEFORE any Gate-0 solve

One standardized 6-panel page per input (**A** distribution · **B** Lorenz with the 30 %-budget
verticals · **C** stopping rule with 3×/5×/10× crossings and the t_min line · **D** universal
transform response with R1 admissibility · **E** spatial top-decile thumbnail, the seam/artifact
check · **F** verdict box with every R-test's actual values) plus a summary sheet (Lorenz overlay,
T2, pre/post-transform leverage bars). **Review each card against the frozen rules before running
the Gate-0 arms** — this is the human checkpoint the protocol builds in.

Note: the spec says "12 cards" but defines the inputs as "8 continuous + EFG block" — that is
**9 pages** (8 + the EFG block card); the miscount is disclosed rather than silently resolved.

In [ ]:
# ---- Feature cards -> audit/feature_cards/ (~15 s for all 10 pages) ----------
card_paths = lc.feature_cards(CARDS)

## Archive — the budget-independent audit objects (spec D2)

Full Lorenz + marginal-density curves per feature, the frozen T2, the constants, and a sha256 of
every input layer. Re-deriving a target at **any θ or any budget** later is interpolation on this
archive — no raster access, no recompute. Lands in `audit/audit_objects/` per the v0.8 directory contract. The demo cell proves it by re-deriving the m_soc target
from the `.npz` alone.

In [ ]:
# ---- Write the archive + prove D2 re-derivation from the archive alone ------
lc.audit_archive(AUDIT_OBJ, tbl)

import json as _json
z = np.load(AUDIT_OBJ / "feature_audit.npz")
cap, ratio = z["irrecoverable_carbon_m_soc__captured"], z["irrecoverable_carbon_m_soc__dens_ratio"]
consts = _json.loads((AUDIT_OBJ / "audit_constants.json").read_text())["constants"]
for th in (10.0, consts["theta"], 3.0):
    ab = ratio >= th
    print(f"  theta = {th:>4.1f}x -> target {float(cap[ab][-1]):.3f}   (lookup on the archive, no raster)")
t_arch = float(cap[ratio >= consts["theta"]][-1])
assert abs(t_arch - 0.332) < 0.005, "archive does not reproduce the frozen target"
print("\nD2 check PASSED: the frozen m_soc target re-derives from feature_audit.npz alone.")

## → Gate 0 arms (the human-in-the-loop hand-off)

The cell below prints the exact `TARGETS`/`WEIGHTS` blocks for `02_solve.ipynb`'s RUN LEVER — one
arm per notebook pass. **All arms use `w = t`** so effective pull (`w/t`) stays 1.00 for every
feature and the *only* thing varying across arms is the stopping point; at `w = 1` a target of
0.332 would raise carbon's pull to 3.0× and its objective-swing share to ~55%, confounding the
comparison. The targets will **bind** (not undershoot) because the control already captures
45.5% / 41.8% at the same pull.

| arm | carbon treatment | why it's in the design |
|---|---|---|
| `a0_control` | none (t = 1.0) | isolates penalty-removal + 1/v; brackets S4 as the extreme carbon-forward |
| `a1_protocol` | m_soc t = 0.332 **only** (biomass reverted by R2) | the protocol's own configuration |
| `a2_flat30` | both pools t = 0.30 | flat comparator at area share |
| `a3_flat40` | both pools t = 0.40 | mild-demotion comparator (biomass binds barely: 41.8% control) |
| `a4_pullcheck` *(optional)* | connectivity at w = t = **0.6** -- UNREACHABLE (cap_max 0.552), so the target can do nothing | empirically proves the w/t invariance claim: must reproduce a0 exactly |

In [ ]:
# ---- Lever-ready blocks, derived from the FROZEN table (not typed by hand) ----
t_msoc = float(row.loc["irrecoverable_carbon_m_soc", "target"])
print("Paste ONE arm into 02_solve.ipynb cell 2 (the RUN LEVER), run it top-to-bottom, repeat:\n")
print(f'''# a0 -- control
RUN <- "a0_control";  TARGETS <- list()

# a1 -- the protocol configuration (from the frozen Gate-0a table; biomass reverted by R2)
RUN <- "a1_protocol"; TARGETS <- list(irrecoverable_carbon_m_soc = {t_msoc})

# a2 -- flat 30%
RUN <- "a2_flat30";   TARGETS <- list(irrecoverable_carbon_m_soc   = 0.30,
                                      irrecoverable_carbon_biomass = 0.30)

# a3 -- flat 40%
RUN <- "a3_flat40";   TARGETS <- list(irrecoverable_carbon_m_soc   = 0.40,
                                      irrecoverable_carbon_biomass = 0.40)

# a4 -- OPTIONAL pull-invariance check (expected: reproduces a0 EXACTLY). t = 0.6 sits ABOVE
#      connectivity cap_max (0.552), so the target is UNREACHABLE and provably inert.
RUN <- "a4_pullcheck"; TARGETS <- list(transboundary_connectivity = 0.6)

WEIGHTS <- TARGETS    # w = t on every arm -- pull 1.00; only the stopping point varies''')
print("\nthen run 03_gate0_validation.ipynb for the verdict tables.")